# Task 5 — Notify (runs when NO new data arrived) — FALSE branch

Runs only when the condition task said FALSE: **zero rows were ingested in the last `window_minutes` minutes**. Prints a clear notification, records an audit row, and the bottom cell explains how REAL email notifications are configured.

In [0]:
# CELL 1 — Parameters + report the situation
dbutils.widgets.text("catalog", "demo1", "Catalog that holds all demo tables")
dbutils.widgets.text("window_minutes", "10", "Look-back window in minutes")
dbutils.widgets.text("job_id", "interactive", "Job ID (passed by the job)")
dbutils.widgets.text("run_id", "interactive", "Run ID (passed by the job)")
dbutils.widgets.text("row_count", "0", "Row count from task validate")

catalog = dbutils.widgets.get("catalog")
window_minutes = dbutils.widgets.get("window_minutes")
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")
row_count = dbutils.widgets.get("row_count")

print("=" * 65)
print("⚠️  NO NEW DATA — notification task executed")
print("=" * 65)
print(f"Rows ingested in the last {window_minutes} minutes : {row_count}")
print("The condition task 'check_row_count' evaluated row_count > 0 as FALSE,")
print("so the silver (SCD1) and gold KPI tasks were SKIPPED and this task ran instead.")

In [0]:
# CELL 2 — Record the notification in the audit log
from datetime import datetime

rows = int(float(row_count)) if row_count.strip() else 0
log_df = spark.createDataFrame(
    [(datetime.now(), job_id, run_id, "notify_no_data", rows, "NO_DATA",
      f"no new rows in the last {window_minutes} minutes — TRUE branch skipped")],
    "logged_at TIMESTAMP, job_id STRING, run_id STRING, task_name STRING, "
    "rows_processed BIGINT, status STRING, message STRING",
)
log_df.write.mode("append").saveAsTable(f"{catalog}.bronze.job_run_log")
print(f"Notification recorded in {catalog}.bronze.job_run_log")

## How REAL email notifications are configured (for the presentation)

This notebook prints a message, but a production job wants actual emails. Databricks Jobs support that natively — **no code needed**:

**Job-level notifications (Jobs UI → your job → Notifications tab, or the Jobs API `email_notifications`):**

* Add recipients — email addresses or workspace groups.
* Choose the events: **job start / success / failure / timeout** (and only for specific tasks).
* This demo job is already configured with `on_failure` → your email — when Scenario C fails, you receive a real failure email.

**Where delivery is controlled:** workspace admins manage available destinations (email, Slack, Microsoft Teams, PagerDuty, webhooks) under **Settings → Notifications**; each user manages their own delivery under **User settings → Notifications**.

**Alert-based notifications (alternative pattern):** create a Databricks SQL **alert** on the bronze table (e.g. "zero rows ingested in the last hour") and subscribe yourself — a data-quality watchdog fully decoupled from the job.

**Job health rules (also configured on this demo job):** under **Health** in the job settings, the rule `RUN_DURATION_SECONDS > 900` marks a run unhealthy and can notify — the 'duration threshold' requirement of this demo.